## Downloading all the dependencies

In [ ]:
!pip install -U langchain
!pip install -U langchain-community
!pip install -U langchain-classic
!pip install -U langchain-text-splitters
!pip install -U langchain-huggingface
!pip install -U langchain-chroma
!pip install -U chromadb
!pip install -U sentence-transformers
!pip install -U transformers
!pip install -U accelerate
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install marker-pdf
!pip install pymupdf


## Importing all the Dependencies

In [ ]:
import fitz  # PyMuPDF
import torch
import gc
import os
import re
import base64
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_classic.storage import LocalFileStore, create_kv_docstore



## Converting the Book to a MarkDown File


In [ ]:

# 1. Load the deep learning models into the GPU ONLY ONCE
print("Loading models into VRAM...")
converter = PdfConverter(
    artifact_dict=create_model_dict(),
)

# Set your file paths
pdf_path = "./docs/Book.pdf" # Make sure this matches your uploaded file name
output_path = "./docs/Book.md"
image_folder = "./extracted_images"

# Create the image folder if it doesn't exist
if not os.path.exists(image_folder):
    os.makedirs(image_folder)

# Open the PDF to calculate pages
doc = fitz.open(pdf_path)
total_pages = len(doc)
batch_size = 50  # Process 50 pages at a time to keep RAM usage extremely low

full_markdown = ""

print(f"Total pages: {total_pages}. Starting batch processing...")

# 2. Loop through the book in chunks of 50 pages
for i in range(0, total_pages, batch_size):
    end_page = min(i + batch_size - 1, total_pages - 1)
    print(f"Processing pages {i} to {end_page}...")

    # Create a temporary PDF for just this batch
    sub_doc = fitz.open()
    sub_doc.insert_pdf(doc, from_page=i, to_page=end_page)
    temp_pdf_path = f"./temp_batch_{i}.pdf"
    sub_doc.save(temp_pdf_path)
    sub_doc.close()

    # 3. Process the small temporary PDF with Marker
    rendered = converter(temp_pdf_path)
    text, _, images = text_from_rendered(rendered)

    # 4. Rename images globally and update the Markdown links
    for original_key, item in images.items():
        # Create a unique global name to prevent overwriting across batches
        new_key = f"batch_{i}_{original_key}"

        # CRITICAL: Find and replace the old image name in the Markdown text
        text = text.replace(original_key, new_key)

        # Save the image to the hard drive with the new unique name
        image_save_path = os.path.join(image_folder, new_key)
        item.save(image_save_path)

    # 5. Append the newly updated text to our master string
    full_markdown += text + "\n\n"

    # 6. CRITICAL: Clean up the RAM and VRAM before the next loop
    os.remove(temp_pdf_path) # Delete the temp PDF
    del rendered, text, images
    gc.collect()             # Force Python to clear System RAM
    if torch.cuda.is_available():
        torch.cuda.empty_cache() # Force PyTorch to clear GPU VRAM

doc.close()

# 7. Save the final stitched markdown to a file
with open(output_path, "w", encoding="utf-8") as f:
    f.write(full_markdown)

print(f"Success! The entire book was converted and saved to {output_path}")
print(f"All images have been saved uniquely to the '{image_folder}' directory.")

## Initializing the Local Embedding Model

In [ ]:
# Initialize BGE-M3 Locally
# This model is a 1024-dimension powerhouse—perfect for medical nuances.


local_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'}, # Use 'cuda' if you have an NVIDIA GPU
    encode_kwargs={'normalize_embeddings': True}
)

toc_mapping = {
    "1": "Introduction to Psychology", "2": "Psychological Research",
    "3": "Biopsychology", "4": "States of Consciousness",
    "5": "Sensation and Perception", "6": "Learning",
    "7": "Thinking and Intelligence", "8": "Memory",
    "9": "Lifespan Developent", "10": "Motivation and Emotion",
    "11": "Personality", "12": "Social Psychology",
    "13": "Industrial-Organizatimonal Psychology", "14": "Stress, Lifestyle, and Health",
    "15": "Psychological Disorders", "16": "Therapy and Treatment"
}

def get_base64_image(img_path):
    if os.path.exists(img_path):
        with open(img_path, "rb") as f:
            return base64.b64encode(f.read()).decode('utf-8')
    return None


print("✅ BGE-M3 Engine and TOC Mapping Ready.")

## Generating the Parent Chunk

In [ ]:
with open("./Book.md", "r", encoding="utf-8") as f:
    markdown_text = f.read()

# Split into Parent Chunks
headers_to_split_on = [("#", "H1"), ("##", "H2"), ("###", "H3"), ("####", "H4")]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)
parent_chunks = markdown_splitter.split_text(markdown_text)

curr_chap, curr_sec, curr_title = "Unknown", "Unknown", "Unknown"

for chunk in parent_chunks:
    # Metadata Tracking Logic
    for h_key in ["H1", "H2", "H3", "H4"]:
        if h_key in chunk.metadata:
            raw = chunk.metadata[h_key]
            clean = re.sub(r'<[^>]+>', '', raw).replace('*', '').strip()
            clean = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', clean) # Clean links
            
            match = re.search(r'^(\d+)\.(\d+)\s+(.*)', clean)
            if match:
                chap_num = match.group(1)
                curr_sec = f"{match.group(1)}.{match.group(2)}"
                curr_title = match.group(3).strip()
                curr_chap = toc_mapping.get(chap_num, "Unknown Chapter")
            elif "CHAPTER OUTLINE" in clean:
                outline_match = re.search(r'\[(\d+)\.\d+\]', chunk.page_content)
                if outline_match:
                    chap_num = outline_match.group(1)
                    curr_chap = toc_mapping.get(chap_num, "Unknown Chapter")
                    curr_sec, curr_title = f"{chap_num}.0", "Introduction"
            del chunk.metadata[h_key]

    # Assign enriched metadata
    chunk.metadata.update({"chapter": curr_chap, "section": curr_sec, "section-title": curr_title})
    
    # Base64 Image Logic
    img_match = re.search(r'!\[\]\((.*?)\)', chunk.page_content)
    if img_match:
        b64 = get_base64_image(img_match.group(1))
        if b64: chunk.metadata['image_base64'] = b64


## Generating the Embedding Locally

In [ ]:
# 1. Setup Persistent Storage as before
persist_dir = "" # Set this to your desired Chroma DB location on Google Drive
parent_store_dir = "" # Set this to your desired Parent Store location on Google Drive

# 2. Create the Parent Storage (This makes the text blocks persistent!)
fs = LocalFileStore(parent_store_dir)
store = create_kv_docstore(fs)

# 3. Setup Chroma as before
vectorstore = Chroma(
    collection_name="med_rag_local",
    embedding_function=local_embeddings,
    persist_directory=persist_dir
)


child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
# 4. Initialize Retriever with the PERSISTENT store
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter
)

# 2. Batching the Parents to stay under the 5461 Child limit
# We use a batch size of 50 parents at a time.
# This roughly generates ~500-800 children per batch, well under the limit.

PARENT_BATCH_SIZE = 50

print(f"🚀 Starting Local Ingestion of {len(parent_chunks)} Parent sections...")

for i in range(0, len(parent_chunks), PARENT_BATCH_SIZE):
    batch = parent_chunks[i : i + PARENT_BATCH_SIZE]
    print(f"Ingesting batch {i//PARENT_BATCH_SIZE + 1}... (Sections {i} to {min(i + PARENT_BATCH_SIZE, len(parent_chunks))})")

    # This call splits the 50 parents into children and embeds them
    retriever.add_documents(batch)

print(f"✅ Ingestion Complete! Data safely saved to {persist_dir}")